# Sesión 11

## Estimación de parámetros en redes bayesianas

> **Objetivos:**
> - Obtener los parámetros de máxima verosimilitud para redes Bayesianas.
> - Obtener los parámetros MAP para redes Bayesianas.

> **Referencias:**
> - Probabilistic Graphical Models: Principles and Techniques, By Daphne Koller and Nir Friedman. Ch. 17.

### 1. Introducción

Una vez que la **estructura** de una red bayesiana está definida, el siguiente paso es **aprender sus parámetros**, es decir, las probabilidades que conforman las *Conditional Probability Distributions* (CPDs) de cada nodo.

Suponemos que disponemos de un conjunto de datos completamente observados:

$$
\mathcal{D} = \{d_1, d_2, \ldots, d_N\}
$$

Cada instancia contiene un valor para cada nodo de la red.  

El objetivo del aprendizaje de parámetros es encontrar los valores de las probabilidades que **mejor explican** los datos observados.

Este problema aparece frecuentemente cuando:
- un experto puede definir la **estructura**, pero no los valores numéricos de las probabilidades,
- queremos ajustar el modelo a **datos reales**,
- necesitamos estos parámetros como parte de procedimientos más complejos (p. ej., aprendizaje de estructura o aprendizaje con datos incompletos).

Existen dos enfoques principales para estimar parámetros:

> **1. Estimación por Máxima Verosimilitud (MLE)**  
> **2. Estimación Bayesiana (MAP / Dirichlet priors)**

#### 1.1 Estimación por Máxima Verosimilitud (MLE)

La idea es simple: elegir los parámetros que **hacen más probable el dataset** bajo el modelo.

Buscamos:
$$
\theta^{*}=\arg\max_\theta P(\mathcal{D}\mid\mathcal{M}(\theta))
$$

Con datos i.i.d.:
$$
P(\mathcal{D}\mid\mathcal{M})=\prod_{m=1}^N P(d_m\mid\mathcal{M})
$$

MLE solo dice: *“ajusta $\theta$ para que lo que viste en los datos sea lo más probable posible”*.

##### 1.1.1. MLE en redes bayesianas discretas

Cada nodo discreto (con o sin padres) genera **conteos** de sus valores observados.

La verosimilitud es la de una **multinomial**: multiplicamos la probabilidad de cada valor **tantas veces como aparece en el dataset**.

Para un nodo con padres $U$:
$$
L(\theta)=\prod_{x,u}\theta_{x\mid u}^{\,N(x,u)}
$$

Para un nodo sin padres:
$$
L(\theta)=\prod_x \theta_x^{\,N(x)}
$$

El máximo siempre se obtiene **normalizando los conteos**:

- Con padres:
  $$\hat\theta_{x\mid u}=\frac{N(x,u)}{N(u)}$$

el estimador de máxima verosimilitud corresponde a la cantidad de muestras consistentes con $x$ y $u$ dividido entre la cantidad de muestras consistentes con $u$.

- Sin padres:
  $$\hat\theta_x=\frac{N(x)}{N}$$

> **MLE = frecuencias relativas.**  
> Porque para datos categóricos, la multinomial dice: *“usa como probabilidad la proporción con la que lo viste ocurrir”*.

##### 1.1.2. Ejemplo 1: nodo **sin padres**

El nodo $X$ toma valores $\{0,1,2\}$. En el dataset observamos:

| $X$ | Conteo |
|-----|--------|
| 0   | 40     |
| 1   | 35     |
| 2   | 25     |

Total:  
$$N=40+35+25=100$$

MLE:
$$
\hat\theta_0=\tfrac{40}{100}=0.40,\quad
\hat\theta_1=\tfrac{35}{100}=0.35,\quad
\hat\theta_2=\tfrac{25}{100}=0.25
$$

##### 1.1.3. Ejemplo 2: nodo **con padres**

Supón que $X$ tiene un padre $U$. Miramos solo las filas donde $U=u$:

| $X$ | $U=u$ | Conteo $N(x,u)$ |
|-----|-------|-----------------|
| 0   |   u   | 30              |
| 1   |   u   | 50              |
| 2   |   u   | 20              |

Total dado $u$:  
$$N(u)=100$$

MLE:
$$
\hat\theta_{0\mid u}=\tfrac{30}{100}=0.30,\quad
\hat\theta_{1\mid u}=\tfrac{50}{100}=0.50,\quad
\hat\theta_{2\mid u}=\tfrac{20}{100}=0.20
$$

> En ambos casos: **frecuencias relativas = parámetros MLE**

#### 1.2. Conexión con el `.fit()` de pgmpy

El método:

``.fit`` de pgmpy implementa exactamente este procedimiento de MLE para redes bayesianas con datos completamente observados.

En este caso vamos a generar un ejemplo de ``Naive Bayes`` usando el dataset de vinos y poniendo atención en la estimación de los parámetros mediante MLE.

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
import os
ruta = os.path.join('..', 'data', 'winequality-red.csv')
wine_data = pd.read_csv(ruta, sep=";")

In [ ]:
wine_data.shape

In [ ]:
wine_data.head()

In [ ]:
# histograma de la variable objetivo
wine_data["quality"].hist(
                        bins=5,
                        color="red",
                        alpha=0.5,
                        width=0.5)

![wine-results](../images/wine.png)

In [ ]:
# target
target = "quality"

# vars independientes
variables = [col for col in wine_data.columns if col != target]

print('Target:', target)
print('Variables independientes:', variables)

In [ ]:
# pandas qcut
pd.qcut?

> **Nota:** En este ejemplo usamos el mismo número de cuantiles para todas las variables solo para simplificar. En la práctica, cada variable debería discretizarse según su propia distribución (no todas requieren la misma cantidad ni el mismo tipo de cortes).

In [ ]:
# Quantización de variables numéricas
quantiles = 10
for col in variables:
    wine_data[col] = pd.qcut(wine_data[col], q=quantiles, labels=range(1, 11))
wine_data.head()

In [ ]:
wine_data['quality'].value_counts().sort_index()

In [ ]:
wine_data['volatile acidity'].value_counts().sort_index()

In [ ]:
# Red Bayesiana
wine_model_naive = DiscreteBayesianNetwork([
    ('quality', 'fixed acidity'),
    ('quality', 'volatile acidity'),
    ('quality', 'citric acid'),
    ('quality', 'residual sugar'),
    ('quality', 'chlorides'),
    ('quality', 'free sulfur dioxide'),
    ('quality', 'total sulfur dioxide'),
    ('quality', 'density'),
    ('quality', 'pH'),
    ('quality', 'sulphates'),
    ('quality', 'alcohol'),
])

> Aquí dejo la [documentación](https://pgmpy.org/plotting.html) acerca de `plotting models` de pgmpy.

In [ ]:
# Dibujar red con daft 
# ojo porque aquí necesitamos instalar daft: https://docs.daft-pgm.org/en/latest/

model_plot = wine_model_naive.to_daft(
    node_pos="shell",
    pgm_params={"observed_style": "outer", "grid_unit": 5}
)
model_plot.render()

In [ ]:
# Train y test
train_df = wine_data.sample(frac=0.8, random_state=42)
test_df = wine_data.drop(train_df.index)
train_df.shape, test_df.shape

* **nota que aquí usamos el `.fit()` de pgmpy para estimar los parámetros de la red bayesiana usando MLE.**

In [ ]:
# Entrenamos
wine_model_naive.fit(train_df)

In [ ]:
wine_model_naive.check_model()

In [ ]:
# CPDs estimadas -- target
print(wine_model_naive.get_cpds(target))

In [ ]:
# train value counts -- target
train_df[target].value_counts(normalize=True).sort_index()

In [ ]:
# CPDs estimadas -- alcohol | target
print(wine_model_naive.get_cpds("alcohol"))

In [ ]:
pd.DataFrame(
    data=wine_model_naive.get_cpds("alcohol").values,
    columns=[f"{target}_{i}" for i in range(3, 9)],
    index=[f"alcohol_{i}" for i in range(1, 11)]
)

In [ ]:
predictions = wine_model_naive.predict(test_df.drop(columns=[target]))

In [ ]:
predictions

In [ ]:
predictions['quality'].values == test_df['quality'].values

In [ ]:
(predictions['quality'].values == test_df['quality'].values).mean()

#### 1.3. `fit_update`

Algo muy interesante del método `fit` de **pgmpy** es que permite un ajuste **incremental** de los parámetros.

_¿Qué significa esto?_  
Que el modelo puede **actualizar sus CPDs con nuevos datos sin volver a entrenar desde cero**. 
 
Es exactamente la idea de aprendizaje **online** o **incremental**: procesar datos que llegan en bloques o secuencias, manteniendo el modelo al día sin reconstruirlo completamente.

Esto es útil cuando:
- El dataset crece con el tiempo.
- Entrenar desde cero sería costoso.
- Queremos modelos que se adapten continuamente.

Aquí está la documentación del método [`fit_update`](https://pgmpy.org/models/bayesiannetwork.html#pgmpy.models.DiscreteBayesianNetwork.DiscreteBayesianNetwork.fit_update).


### 2. Estimación MAP

#### 2.1. ¿Cómo prevenimos el _overfitting_?

MLE copia **exactamente** las frecuencias del dataset.
Si el dataset es pequeño o hay combinaciones poco frecuentes (tienden a cero) → las probabilidades quedan **extremas** y el modelo **sobreajusta**.

**Solución:** introducir *suavizado* mediante **regularización / priors** (p. ej., Dirichlet). Esto evita ceros, reparte _probabilidad_ antes de ver datos y produce parámetros más estables.

##### 2.1.1 ¿Qué es la prior Dirichlet?

Queremos modelar una distribución multinomial con parámetros $\bar{\theta} = [\theta_1,\dots,\theta_k]$.
Estas $\theta_i$ son **probabilidades**, así que deben sumar 1 y ser ≥ 0.

_Dirichlet_ es simplemente una **distribución sobre vectores de probabilidad**. 

Es decir:

> La Dirichlet describe “cómo creemos que deberían ser las probabilidades” antes de ver los datos.


##### 2.1.2 ¿Qué significan sus parámetros?

La Dirichlet tiene hiperparámetros $\alpha_1,\dots,\alpha_k$ y su interpretación intuitiva es:

- Cada $\alpha_i$ funciona como **cuántas veces creo que ocurrió la categoría $i$ antes de ver datos reales**.
- Por eso se conocen como **conteos ficticios** _(pseudo-counts)_.

Ejemplo:  

si $\alpha = [1,1,1]$ → no tengo preferencias (todo es uniforme).

si $\alpha = [10,1,1]$ → creo fuertemente que la categoría 1 es más probable.



##### 2.1.3 ¿Cómo actualiza la Dirichlet con los datos?

Si contamos en los datos reales cuántas veces apareció cada categoría $i$: 

- $N_i$ = conteos reales

Entonces la actualización posterior es:

$$
\hat\theta_i = \frac{N_i + \alpha_i}{\sum_j (N_j + \alpha_j)}
$$

Es decir:

> **Probabilidad = (conteos reales + conteos ficticios) / total.**

Muy simple:  
- Los datos suman evidencia.  
- La prior agrega “un pequeño empujón” que evita ceros y reduce overfitting.

> **Dirichlet = multinomial suavizada.**  
>  
> Toma las frecuencias reales y les agrega un poquito de evidencia previa para no sobreajustar.

Es exactamente lo que necesitas cuando:
- tienes pocos datos,
- aparecen valores raros,
- quieres evitar probabilidades 0,
- quieres un modelo más generalizable.

- Estimadores MAP son menos sensibles al ruido en los datos.

- Estimador MAP $\to$ estimador de máxima verosimilitud cuando el número de muestras $\to \infty$.

In [ ]:
from pgmpy.estimators import BayesianEstimator

In [ ]:
help(BayesianEstimator)

In [ ]:
estimator = BayesianEstimator(
    model=wine_model_naive,
    data=train_df
)

##### 2.1.4. Los _pseudo-counts_ (los $\alpha$ de Dirichlet)

* **Cada nodo tiene 6 categorías**

In [ ]:
train_df[target].nunique()

In [ ]:
np.ones((train_df[target].nunique(), 1))

* **10 es un hiperparámetro que define la fuerza del prior Dirichlet.**

In [ ]:
np.ones((train_df[target].nunique(), 1)) * 10

Supongamoss que los conteos reales fueron:

| Categoría | Conteo real $N_i$ |
|-----------|-------------------|
| 0         | 50                |
| 1         | 30                |
| 2         | 20                |
| 3         | 10                |
| 4         | 5                 |
| 5         | 1                 |

Prior Dirichlet utilizado es:

$$
\alpha_i = 10 \quad \forall i
$$

La actualización posterior es:

$$
\alpha'_i = N_i + 10
$$

Y la CPD resultante (la media posterior del Dirichlet) es:

$$
P(i) = \frac{N_i + 10}{\sum_j (N_j + 10)}
$$

Esto equivale a suponer que:

> Antes de ver datos, he observado 10 veces cada categoría

In [ ]:
# Estimar CPDs con previa de Dirichlet

cpd_quality = estimator.estimate_cpd(
    node=target,
    prior_type="dirichlet",
    pseudo_counts=10 * np.ones((train_df[target].nunique(), 1))
)
print(cpd_quality)

In [ ]:
#para comparar
print(wine_model_naive.get_cpds(target))

In [ ]:
cpd_cols = {}
for col in variables:
    cpd_cols[col] = estimator.estimate_cpd(
        node=col,
        prior_type="dirichlet",
        pseudo_counts=10 * np.ones((train_df[col].nunique(), train_df[target].nunique()))
    )
print(cpd_cols[col])

* **¿Por qué todos los $\alpha_i$ son iguales?**

> Porque queremos un prior **no informativo** que no favorezca ninguna categoría en particular.

En este caso estamos usando un prior **simétrico** (todos los $\alpha_i$ iguales) que no favorece ninguna categoría.

Sin embargo, podríamos usar **priors asimétricos** si tuviéramos conocimiento previo que favoreciera ciertas categorías.

Por ejemplo:

```python
pseudo_counts = np.array([[20],[10],[5],[2],[1],[1]])
```

* **Como usualmente hacemos, agregamos o sumamos las cpds al modelo:**

In [ ]:
# Añadimos CPDs
wine_model_naive.add_cpds(cpd_quality, * cpd_cols.values())
wine_model_naive.check_model()

* **Generamos las predicciones:**

In [ ]:
predictions = wine_model_naive.predict(test_df.drop(columns=[target]))

In [ ]:
predictions

In [ ]:
predictions['quality'].values == test_df['quality'].values

In [ ]:
# accuracy
(predictions['quality'].values == test_df['quality'].values).mean()